# W&B Reports API — Publish a Hodges-framed HiLiftAeroML narrative as a Report

This notebook builds a W&B Report whose section structure follows
Justin Hodges' HiLiftAeroML tutorial
(`docs/explainers/surrogate_ml_hodges.pdf`), with prose in the
workshop's own voice that summarises and points at the PDF for the
original text. Hodges' Notebook 1 (0D KPI) section is dropped since
the workshop only covers the 3D point-cloud surrogate. The Report
ends with a `PanelGrid` showing the 2D prediction triptychs logged
by `experiment_tracking_hilift.ipynb` at each cadence epoch — the
narrative lands the reader on a live view of a real training run's
output.

**Prerequisites**
- `uv sync` (adds `wandb-workspaces` to your environment).
- `uv run wandb login`.
- `experiment_tracking_hilift.ipynb` has been executed end-to-end at
  least once, so `wandb_emea/jb_physics_demo` contains a
  `hilift-transolverpp-train` run whose `val_plots/predictions_2d_epoch_*`
  media entries the panel below queries.


## 1. Setup

Imports and the target project. `wandb-workspaces` is a separate
package that wraps the Reports REST API; the workspaces module is
imported as `wr` by convention.


In [1]:
import wandb
import wandb_workspaces.reports.v2 as wr

ENTITY = "wandb_emea"
PROJECT = "jb_physicsx_demo"

# The training run whose 2D plots we surface at the end.
RUN_QUERY = "hilift-transolverpp-train"

print(f"wandb            = {wandb.__version__}")
print(f"target project   = {ENTITY}/{PROJECT}")
print(f"runset query     = {RUN_QUERY}")


wandb            = 0.28.0
target project   = wandb_emea/jb_physicsx_demo
runset query     = hilift-transolverpp-train


## 2. Build the Report

The blocks below reproduce Hodges' tutorial section-by-section.
Prose is close to verbatim — the only structural edit is dropping
his Notebook 1 (0D KPI) section, since this workshop's notebook
implements only the 3D point-cloud example. A single `PanelGrid` at
the end surfaces the 2D prediction plots logged by the training
run.


In [2]:
# ---- Runset for the training run at the end of the Report ----
hilift_runset = wr.Runset(
    entity=ENTITY,
    project=PROJECT,
    name="HiLift Transolver++ training",
    query=RUN_QUERY,
)

# ---- PanelGrid: 2D prediction triptychs across training ----
predictions_2d_grid = wr.PanelGrid(
    runsets=[hilift_runset],
    panels=[
        wr.MediaBrowser(
            media_keys=[
               # "val_plots/predictions_2d_epoch_10",
                "val_plots/predictions_2d_epoch_20",
                # "val_plots/predictions_2d_epoch_30",
                # "val_plots/predictions_2d_epoch_40",
                # "val_plots/predictions_2d_epoch_50",
            ],
            num_columns=1,
        ),
    ],
)


In [3]:
blocks = [
    # ---- Title + framing + attribution ----
    wr.H1(text="HiLiftAeroML — a beginner's guide with a live surrogate run"),
    wr.MarkdownBlock(text="*Structured after Justin Hodges' HiLiftAeroML tutorial.*"),
    wr.MarkdownBlock(text=(
        "This Report follows the arc of Justin Hodges' tutorial "
        "*A Beginner's Tutorial: the new HiLiftAeroML dataset with "
        "code to train a ML surrogate* (May 2026) — see the original "
        "for the full text. Hodges' Notebook 1 (a 0D KPI regressor "
        "over CL/CD/CM) is skipped here because the workshop only "
        "implements his Notebook 2, the 3D point-cloud surrogate. "
        "The narrative below is a workshop-voiced summary; live "
        "prediction plots from a real training run appear at the "
        "end. Source: `docs/explainers/surrogate_ml_hodges.pdf`."
    )),

    # ---- Opening framing (Hodges' page 1-2 intro) ----
    wr.MarkdownBlock(text=(
        "HiLiftAeroML is a newly-open aerodynamics dataset built on "
        "the NASA CRM-HL high-lift wing. Hodges frames it as "
        "unusually important for ML surrogate work because it "
        "brings high-fidelity, open CFD data to a problem class "
        "that has, until now, been studied mainly through the AIAA "
        "CFD High-Lift Prediction Workshop series and closely "
        "related validation campaigns — solver comparisons, "
        "turbulence-model studies, mesh-sensitivity work. CRM-HL "
        "itself is the standardised wing geometry those workshops "
        "have been built around.\n\n"
        "The change HiLiftAeroML introduces, per Hodges, is scale + "
        "openness: **1,800 high-fidelity WMLES simulations across "
        "180 geometry variants × 10 angles of attack**, packaged "
        "with forces, geometry definitions, surface fields, volume "
        "fields, and visual diagnostics under an open licence. "
        "Before this release, the CRM-HL ecosystem was mostly a CFD "
        "validation and methods-development platform — not a "
        "training corpus.\n\n"
        "Hodges walks through two examples in the tutorial (a 0D "
        "coefficient regressor and a 3D point-cloud field "
        "surrogate). **This Report covers only the 3D case**, "
        "because that's what the workshop's "
        "`experiment_tracking_hilift.ipynb` implements."
    )),

    # ---- The Dataset ----
    wr.H2(text="The Dataset"),
    wr.MarkdownBlock(text=(
        "Per Hodges, HiLiftAeroML publishes each CRM-HL case as a "
        "mix of small tabular summaries and very large field "
        "archives:\n\n"
        "- `force_mom_all.csv` — integrated lift, drag, moment, "
        "pressure, and viscous coefficients.\n"
        "- `geo_values_all.csv` — the geometry design variables.\n"
        "- `boundary_*.vtu.tgz` (per run) — time-averaged surface "
        "fields.\n"
        "- `volume_*.vtu.tgz` (per run) — time-averaged volume "
        "fields.\n"
        "- STL/STP geometry files and rendered wall-shear "
        "visualisations.\n\n"
        "The practical constraint Hodges flags is **file size**. "
        "The coefficient CSVs are trivial to work with. The field "
        "archives are not — the dataset card lists boundary "
        "archives at roughly **13 GB compressed per case** and "
        "volume archives at roughly **23 GB per case**. That "
        "constraint drives the design of every beginner example: "
        "you can't just download the whole thing and iterate."
    )),

    # ---- Model Choice: Why Transolver++ ----
    wr.H2(text="Model Choice: Why Transolver++"),
    wr.MarkdownBlock(text=(
        "Hodges picks Transolver++ — a neural PDE-solver "
        "architecture aimed at large, irregular geometries — "
        "because CFD data lives on meshes and point clouds, not on "
        "image grids. Its **physics-attention** mechanism "
        "aggregates information from many mesh points into a small "
        "set of learned physical states, and expresses per-point "
        "predictions as a soft combination of those states. That's "
        "a natural fit for surface and volume CFD fields.\n\n"
        "Hodges notes explicitly that other point-cloud / graph / "
        "mesh models would also work; Transolver++ is one "
        "reasonable choice, not the only one. The workshop keeps "
        "the model deliberately small so it fits on a laptop CPU. "
        "See `docs/explainers/pxtransolver.md` for the "
        "attention-mechanism walkthrough."
    )),

    # ---- 3D Point-Cloud Prediction (Hodges' Notebook 2) ----
    wr.H2(text="3D Point-Cloud Prediction"),
    wr.MarkdownBlock(text=(
        "This is the piece of Hodges' tutorial the workshop "
        "implements. The task moves from global coefficients to "
        "**per-point prediction of a scalar field over a sampled "
        "surface point cloud** — closer to what most people mean "
        "by an ML-CFD surrogate.\n\n"
        "The training-data shape is:\n\n"
        "- **inputs:** sampled surface points + surface normals + "
        "AoA condition\n"
        "- **outputs:** scalar field value at each point (e.g. Cp "
        "or wall-shear magnitude)\n\n"
        "The serious data path (Hodges' recommendation) reads a "
        "`boundary_*.vtu.tgz` archive, samples surface points, "
        "pulls coordinates and normals, picks a target field, and "
        "caches the reduced point cloud as a compact `.npz`. But "
        "because each raw archive is ~13 GB, Hodges also ships a "
        "**demo point-cloud mode** — small synthetic pressure-like "
        "point clouds that keep the training and visualisation "
        "pipeline runnable on modest hardware. The workshop's "
        "`registry.ipynb` implements a workshop-tuned version of "
        "that generator (slat / main-wing / flap topology in the "
        "CRM-HL proportions) and publishes it to the Registry.\n\n"
        "The visual payoff — versus the coefficient regressor — is "
        "that you get to inspect predictions **spatially**: "
        "ground-truth surface field, predicted field, absolute "
        "error over the same geometry, and 2D projected views for "
        "easier scanning. The panel at the bottom of this Report "
        "surfaces those 2D views live from the workshop's training "
        "run."
    )),

    # ---- What This Example Is, and Is Not ----
    wr.H2(text="What This Example Is, and Is Not"),
    wr.MarkdownBlock(text=(
        "Hodges is explicit that this is a **beginner-scale** "
        "example on purpose. The point-cloud path downsamples "
        "heavily, starts with a small number of cases, uses a "
        "Transolver++ configuration that fits on Colab, and writes "
        "training loops for clarity rather than performance. "
        "Making it industrial-grade — real VTUs, many cases, big "
        "GPUs, hours-long training — would push the tutorial out "
        "of beginner range.\n\n"
        "The right way to read the panels below is as a **pipeline "
        "shake-out**: does the training loop run, do the metrics "
        "descend, do the visualisations render, does the "
        "Registry-consumption pattern work end-to-end? Those are "
        "the workshop's questions. Aerodynamic validity of the "
        "synthetic target is not one of them — for that you'd need "
        "real HiLiftAeroML boundary fields and a real held-out "
        "geometry."
    )),

    # ---- Making It Serious ----
    wr.H2(text="Making It Serious"),
    wr.MarkdownBlock(text=(
        "The path Hodges sketches from beginner to serious is a "
        "checklist:\n\n"
        "- Swap the demo generator for a VTU reader over real "
        "`boundary_*.vtu.tgz` cases.\n"
        "- Scale case count into the O(100+) regime and split by "
        "geometry rather than by row, so the model has to "
        "generalise to unseen aircraft variants.\n"
        "- If you're benchmarking, follow the dataset's official "
        "train/val/test splits.\n"
        "- Predict physically meaningful fields — Cp, wall-shear "
        "magnitude, or several channels jointly.\n"
        "- Recover integrated CL / CD / CM from predicted pressure "
        "or shear fields and cross-check against "
        "`force_mom_all.csv`.\n"
        "- Add strong baselines: MLP, PointNet, graph neural "
        "network, MeshGraphNet-style model, or classical "
        "interpolation. Transolver++ should sit in a table, not by "
        "itself.\n"
        "- Evaluate geometry generalisation and AoA interpolation "
        "separately — they fail in different ways.\n"
        "- Track error **by aircraft component**: slat, main wing, "
        "flap, tip, junction. Aggregate metrics hide the "
        "interesting failure modes.\n"
        "- Add uncertainty estimates or ensembles, especially near "
        "stall and separated flow.\n"
        "- Build a reusable VTU-to-point-cloud preprocessing "
        "pipeline; it will pay off on every subsequent experiment.\n\n"
        "Hodges' framing for portfolio value: the strong version "
        "of this project isn't *\"I trained a model on a CFD "
        "dataset\"* but something more specific — a Transolver++ "
        "point-cloud surrogate for CRM-HL high-lift CFD, with "
        "VTU preprocessing, geometry-conditioned training, "
        "surface-field visualisation, and coefficient recovery "
        "from predicted pressure fields. For publishable work he "
        "wants systematic scope: curated benchmark, reproducible "
        "splits, strong baselines, field-level and "
        "coefficient-level metrics, careful failure analysis."
    )),

    # ---- 2D prediction plots panel at the very end ----
    wr.H2(text="2D prediction plots from the workshop training run"),
    wr.MarkdownBlock(text=(
        "The panel below queries live from the "
        "`hilift-transolverpp-train` run in "
        "`wandb_emea/jb_physics_demo`. It stacks the 2D top-view "
        "prediction triptychs (`val_plots/predictions_2d_epoch_*`) "
        "logged at every cadence epoch (10, 20, 30, 40, 50). Each "
        "triptych shows CFD truth, Transolver++ prediction, and "
        "absolute error on the same canonical test case, so "
        "reading top-to-bottom is a progression view of how the "
        "prediction sharpens over training."
    )),
    predictions_2d_grid,
]


## 3. Save and share

`report.save()` publishes the Report to the target project as a draft.
`report.url` returns the shareable link — the same URL you would get
from clicking **Create report** in the W&B UI.


In [4]:
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="HiLiftAeroML — a beginner's guide with a live surrogate run",
    description=(
        "Follows the section arc of Justin Hodges' HiLiftAeroML "
        "tutorial (May 2026) with workshop-voiced summaries; skips "
        "Hodges' Notebook 1 (0D KPI) section. Ends with the 2D "
        "prediction plots from the workshop's "
        "`hilift-transolverpp-train` run."
    ),
    blocks=blocks,
)
report.save()

print(report.url)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Saved report to: https://wandb.ai/wandb_emea/jb_physicsx_demo/reports/HiLiftAeroML-—-a-beginner's-guide-with-a-live-surrogate-run--VmlldzoxNzUwMTQ3Nw==


https://wandb.ai/wandb_emea/jb_physicsx_demo/reports/HiLiftAeroML-—-a-beginner's-guide-with-a-live-surrogate-run--VmlldzoxNzUwMTQ3Nw==
